# GroupDNA: WhatsApp Chat Analyzer
**Name:** Priyam Bidhan
**Date:** June 2026

*A Python tool to analyze WhatsApp group chats using only fundamentals and NumPy.*


In [1]:
import numpy as np
from datetime import datetime, timedelta
import string
try:
    from google.colab import files
    print("Please upload your WhatsApp chat .txt file:")
    uploaded = files.upload()
    FILE_PATH = list(uploaded.keys())[0]
    GROUP_NAME = FILE_PATH.replace('.txt', '').replace('WhatsApp Chat with ', '')
except ImportError:
    FILE_PATH = 'hostel_bois.txt'
    GROUP_NAME = 'Hostel Bois 4ever'
# Helper function to handle various WhatsApp timestamp formats
def parse_timestamp(ts_str):
    ts_str = ts_str.replace('\u202f', ' ').replace('\u200e', '').upper()
    formats = [
        '%d/%m/%y, %H:%M',
        '%d/%m/%y, %I:%M %p',
        '%d/%m/%Y, %H:%M',
        '%d/%m/%Y, %I:%M %p',
        '%m/%d/%y, %H:%M',
        '%m/%d/%y, %I:%M %p',
        '%y/%m/%d, %H:%M',
        '%Y/%m/%d, %H:%M',
    ]
    for fmt in formats:
        try:
            return datetime.strptime(ts_str, fmt)
        except ValueError:
            continue
    try:
        return datetime.strptime(ts_str.split(',')[0], '%d/%m/%y')
    except:
        pass
    return None


Please upload your WhatsApp chat .txt file:


Saving hostel_bois.txt to hostel_bois.txt


## Feature 1: The Chat Parser
Reads the chat file line by line, handling edge cases (system messages, media, deleted messages, multi-line) and extracts timestamp, sender, and text.


In [2]:
def parse_chat(file_path):
    messages = []
    sys_count = media_count = del_count = 0
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"File {file_path} not found. Please upload it.")
        return [], 0, 0, 0
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if len(line) >= 8 and '/' in line[:5]:
            parts = line.split(' - ', 1)
            if len(parts) != 2:
                continue
            timestamp_str, rest = parts[0], parts[1]
            dt_obj = parse_timestamp(timestamp_str)
            if dt_obj is None:
                if messages: messages[-1]['text'] += ' ' + line
                continue
            if ': ' not in rest:
                sys_count += 1
                continue
            sender, text = rest.split(': ', 1)
            if text == '<Media omitted>':
                media_count += 1
                messages.append({'ts': timestamp_str, 'dt_obj': dt_obj, 'sender': sender, 'text': text, 'valid': False})
            elif text == 'This message was deleted':
                del_count += 1
                messages.append({'ts': timestamp_str, 'dt_obj': dt_obj, 'sender': sender, 'text': text, 'valid': False})
            else:
                messages.append({'ts': timestamp_str, 'dt_obj': dt_obj, 'sender': sender, 'text': text, 'valid': True})
        else:
            if messages:
                messages[-1]['text'] += ' ' + line

    real_count = sum(1 for m in messages if m['valid'])
    print(f"Successfully parsed {real_count} real messages, skipped {sys_count} system messages, {media_count} media-omitted, {del_count} deleted messages.")
    return messages, sys_count, media_count, del_count
messages, sys_count, media_count, del_count = parse_chat(FILE_PATH)


Successfully parsed 3127 real messages, skipped 4 system messages, 32 media-omitted, 15 deleted messages.


## Feature 2 & 3: Group Overview and Most Active Day/Hour


In [3]:
def get_overview_stats(messages):
    real_msgs = [m for m in messages if m['valid']]
    if not real_msgs: return {}
    d1 = real_msgs[0]['dt_obj'].date()
    d2 = real_msgs[-1]['dt_obj'].date()
    total_days = (d2 - d1).days + 1
    if total_days <= 0: total_days = 1
    person_counts = {}
    date_counts = {}
    hour_counts = {}
    for m in real_msgs:
        sender = m['sender']
        person_counts[sender] = person_counts.get(sender, 0) + 1
        date_str = m['dt_obj'].strftime('%d %B %Y')
        hour_val = m['dt_obj'].hour
        date_counts[date_str] = date_counts.get(date_str, 0) + 1
        hour_counts[hour_val] = hour_counts.get(hour_val, 0) + 1
    sorted_persons = sorted(person_counts.items(), key=lambda x: x[1], reverse=True)
    busiest_date = max(date_counts.items(), key=lambda x: x[1])
    busiest_hour = max(hour_counts.items(), key=lambda x: x[1])
    return {
        'start_date': d1.strftime('%d %B %Y'),
        'end_date': d2.strftime('%d %B %Y'),
        'total_days': total_days,
        'total_msgs': len(real_msgs),
        'participants': sorted_persons,
        'busiest_date_str': busiest_date[0],
        'busiest_date_count': busiest_date[1],
        'busiest_hour': busiest_hour[0],
        'busiest_hour_count': busiest_hour[1]
    }
if messages:
    stats = get_overview_stats(messages)


## Feature 4: Activity Heatmap (NumPy)


In [4]:
def build_heatmap(messages, stats):
    real_msgs = [m for m in messages if m['valid']]
    ordered_participants = [p for p, _ in stats['participants']]
    person_idx = {p: i for i, p in enumerate(ordered_participants)}
    heatmap = np.zeros((len(ordered_participants), 24), dtype=int)
    for m in real_msgs:
        hour = m['dt_obj'].hour
        heatmap[person_idx[m['sender']], hour] += 1
    return heatmap, ordered_participants
if messages:
    heatmap_matrix, heatmap_participants = build_heatmap(messages, stats)

## Feature 5: Top Words


In [5]:
def get_top_words(messages):
    real_msgs = [m for m in messages if m['valid']]
    stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'my', 'you', 'that', 'this', 'are', 'hai', 'bhi', 'se', 'ko', 'ki', 'me', 'ho', 'ke', 'ka', 'toh', 'ye'}
    word_counts = {}
    for m in real_msgs:
        text = m['text'].lower()
        for p in string.punctuation:
            text = text.replace(p, '')
        for w in text.split():
            if w not in stop_words and len(w) > 1:
                word_counts[w] = word_counts.get(w, 0) + 1
    return sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]
if messages:
    top_words = get_top_words(messages)


## Feature 6: Response Speed & Silent Streaks


In [6]:
def get_response_and_streaks(messages, stats):
    real_msgs = [m for m in messages if m['valid']]
    participants = [p for p, _ in stats['participants']]
    response_gaps = {p: [] for p in participants}
    for i in range(1, len(real_msgs)):
        curr, prev = real_msgs[i], real_msgs[i-1]
        if curr['sender'] != prev['sender']:
            gap = (curr['dt_obj'] - prev['dt_obj']).total_seconds()
            if gap >= 0:  # Prevent negative gaps if timestamps are weird
                response_gaps[curr['sender']].append(gap)
    avg_responses = {}
    for p, gaps in response_gaps.items():
        avg_responses[p] = sum(gaps) / len(gaps) if gaps else 0
    person_days = {p: set() for p in participants}
    for m in real_msgs:
        person_days[m['sender']].add(m['dt_obj'].date())
    start_date = real_msgs[0]['dt_obj'].date()
    end_date = real_msgs[-1]['dt_obj'].date()
    streaks = {}
    for p, active in person_days.items():
        max_streak = curr_streak = 0
        best_start = best_end = streak_start = None
        curr = start_date
        while curr <= end_date:
            if curr not in active:
                if curr_streak == 0: streak_start = curr
                curr_streak += 1
                if curr_streak > max_streak:
                    max_streak = curr_streak
                    best_start, best_end = streak_start, curr
            else:
                curr_streak = 0
            curr += timedelta(days=1)
        streaks[p] = {'max': max_streak, 'start': best_start, 'end': best_end, 'active_days': len(active)}
    return avg_responses, streaks
if messages:
    responses, streaks = get_response_and_streaks(messages, stats)

## Feature 7: Personality Archetype Detection


In [7]:
def detect_archetypes(messages, stats, streaks):
    real_msgs = [m for m in messages if m['valid']]
    participants = [p for p, _ in stats['participants']]
    scores = {p: {} for p in participants}
    # 1. THE SPAMMER
    bursts = {p: [] for p in participants}
    curr_s, curr_b = real_msgs[0]['sender'], 1
    for i in range(1, len(real_msgs)):
        if real_msgs[i]['sender'] == curr_s: curr_b += 1
        else:
            bursts[curr_s].append(curr_b)
            curr_s, curr_b = real_msgs[i]['sender'], 1
    bursts[curr_s].append(curr_b)
    for p in participants: scores[p]['THE SPAMMER'] = sum(bursts[p])/len(bursts[p]) if bursts[p] else 0
    # 2. THE GROUP MOM
    kws = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
    for p in participants:
        scores[p]['THE GROUP MOM'] = sum(1 for m in real_msgs if m['sender'] == p and any(kw in m['text'].lower() for kw in kws))

    # 3. THE NIGHT OWL
    for p in participants:
        p_msgs = [m for m in real_msgs if m['sender'] == p]
        scores[p]['THE NIGHT OWL'] = (sum(1 for m in p_msgs if m['dt_obj'].hour >= 23 or m['dt_obj'].hour < 5) / len(p_msgs) * 100) if p_msgs else 0
    # 4. THE STORYTELLER
    for p in participants:
        p_msgs = [m['text'] for m in real_msgs if m['sender'] == p]
        scores[p]['THE STORYTELLER'] = sum(len(t.split()) for t in p_msgs) / len(p_msgs) if p_msgs else 0
    # 5. THE DRAMA QUEEN
    for p in participants:
        p_msgs = [m['text'] for m in real_msgs if m['sender'] == p]
        dq = sum(1 for t in p_msgs if (len(''.join(c for c in t if c.isalpha())) >= 3 and ''.join(c for c in t if c.isalpha()).isupper()) or t.count('!') >= 2)
        scores[p]['THE DRAMA QUEEN'] = (dq / len(p_msgs) * 100) if p_msgs else 0
    # 6. THE GHOST
    for p in participants:
        scores[p]['THE GHOST'] = ((stats['total_days'] - streaks[p]['active_days']) / stats['total_days']) * 100
    # 7. THE COMEDIAN
    c_kws = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
    for p in participants:
        p_msgs = [m['text'].lower() for m in real_msgs if m['sender'] == p]
        scores[p]['THE COMEDIAN'] = (sum(1 for t in p_msgs if any(kw in t for kw in c_kws)) / len(p_msgs) * 100) if p_msgs else 0
    # 8. THE QUESTION MASTER
    for p in participants:
        p_msgs = [m['text'].strip() for m in real_msgs if m['sender'] == p]
        scores[p]['THE QUESTION MASTER'] = (sum(1 for t in p_msgs if t.endswith('?')) / len(p_msgs) * 100) if p_msgs else 0
    # 9. BONUS: THE WEEKEND WARRIOR (Custom Archetype)
    for p in participants:
        p_msgs = [m for m in real_msgs if m['sender'] == p]
        scores[p]['THE WEEKEND WARRIOR'] = (sum(1 for m in p_msgs if m['dt_obj'].weekday() >= 5) / len(p_msgs) * 100) if p_msgs else 0

    # Assignment Logic (Normalize across people, assign highest remaining)
    arch_list = list(scores[participants[0]].keys())
    norm = {p: {} for p in participants}
    for a in arch_list:
        m_val = max(scores[p][a] for p in participants)
        for p in participants: norm[p][a] = scores[p][a] / m_val if m_val > 0 else 0

    flat = sorted([(p, a, norm[p][a], scores[p][a]) for p in participants for a in arch_list], key=lambda x: x[2], reverse=True)

    assignments, details = {}, {}
    used_a = set()
    for p, a, _, raw in flat:
        if p not in assignments and a not in used_a:
            assignments[p] = a
            used_a.add(a)
            if a == 'THE SPAMMER': details[p] = f"(avg {raw:.1f} msgs in a row)"
            elif a == 'THE GROUP MOM': details[p] = f"(caring keyword score: {raw})"
            elif a == 'THE NIGHT OWL': details[p] = f"({raw:.1f}% msgs between 23h-04h)"
            elif a == 'THE STORYTELLER': details[p] = f"(avg {raw:.1f} words per msg)"
            elif a == 'THE DRAMA QUEEN': details[p] = f"({raw:.1f}% ALL-CAPS messages)"
            elif a == 'THE GHOST': details[p] = f"(silent on {int(raw/100 * stats['total_days'])} of {stats['total_days']} days)"
            elif a == 'THE COMEDIAN': details[p] = f"({raw:.1f}% haha/lol msgs)"
            elif a == 'THE QUESTION MASTER': details[p] = f"({raw:.1f}% msgs ending in ?)"
            elif a == 'THE WEEKEND WARRIOR': details[p] = f"({raw:.1f}% msgs on weekends)"

    # Fallback for large groups: if there are more than 9 people, some won't get a unique archetype.
    for p in participants:
        if p not in assignments:
            assignments[p] = 'THE NORMIE'
            details[p] = '(just chilling)'

    return assignments, details

if messages:
    archs, arch_details = detect_archetypes(messages, stats, streaks)


## Feature 8: The Final Report


In [8]:
def format_time(sec):
    return f"{sec/60:.1f} minutes" if sec < 3600 else f"{sec/3600:.1f} hours"

def print_final_report():
    print("============================================================")
    print(f" GROUPDNA REPORT — \"{GROUP_NAME}\"")
    print(f" {stats['total_days']} days • {stats['total_msgs']:,} messages • {len(stats['participants'])} members")
    print("============================================================")

    print(f"\n Period             : {stats['start_date']} to {stats['end_date']}")
    print(f" Busiest day        : {stats['busiest_date_str']} ({stats['busiest_date_count']} messages)")
    h = stats['busiest_hour']
    print(f" Busiest hour       : {h:02d}:00 - {h+1:02d}:00")

    print("\n MESSAGES PER PERSON")
    for p, c in stats['participants']:
        bar = '█' * int((c / stats['participants'][0][1]) * 20)
        pct = (c / stats['total_msgs']) * 100
        print(f" {p:<10} {bar:<20} {c:>4} ({pct:>4.1f}%)")

    print("\n ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
    header = "".join([f"{h:02d}" if h % 3 == 0 else "  " for h in range(24)])
    print(f"           {header}")

    for p in heatmap_participants:
        row = heatmap_matrix[heatmap_participants.index(p)]
        m_val = np.max(row) if np.max(row) > 0 else 1
        blocks = ""
        for val in row:
            r = val / m_val
            if r > 0.75: blocks += "█ "
            elif r > 0.50: blocks += "▒ "
            elif r > 0.25: blocks += "░ "
            else: blocks += ". "

        # Add '<- NIGHT OWL' tag if applicable
        tag = " <- NIGHT OWL" if archs.get(p) == "THE NIGHT OWL" else ""
        print(f" {p:<10}{blocks}{tag}")

    print("\n THIS GROUP'S FAVOURITE WORDS")
    for w, c in top_words:
        bar = '█' * int((c / top_words[0][1]) * 20)
        print(f" {w:<10} {bar:<20} {c}")

    print("\n RESPONSE PATTERNS")
    sorted_resp = sorted(responses.items(), key=lambda x: x[1])
    print(f" Fastest replier : {sorted_resp[0][0]} (avg {format_time(sorted_resp[0][1])})")
    print(f" Slowest replier : {sorted_resp[-1][0]} (avg {format_time(sorted_resp[-1][1])})")

    print("\n LONGEST SILENT STREAKS")
    sorted_streaks = sorted(streaks.items(), key=lambda x: x[1]['max'], reverse=True)
    for p, s in sorted_streaks[:2]:
        if s['max'] > 0:
            print(f" {p:<10} : {s['max']} days ({s['start'].strftime('%d %b')} - {s['end'].strftime('%d %b')})")
        else:
            print(f" {p:<10} : 0 days")

    for p, s in sorted_streaks[2:]:
        if s['max'] > 0:
            print(f" {p:<10} : {s['max']} days")
        else:
            print(f" {p:<10} : 0 days")

    print("\n PERSONALITY ARCHETYPES")
    for p in heatmap_participants:
        print(f" {p:<10} → {archs[p]:<18} {arch_details[p]}")

    print("\n============================================================")
    print(" Generated by GroupDNA • Built with Python + NumPy")
    print("============================================================")

if messages:
    print_final_report()


 GROUPDNA REPORT — "hostel_bois"
 60 days • 3,127 messages • 6 members

 Period             : 01 April 2024 to 30 May 2024
 Busiest day        : 04 May 2024 (74 messages)
 Busiest hour       : 18:00 - 19:00

 MESSAGES PER PERSON
 Rahul      ████████████████████  940 (30.1%)
 Priya      ███████████████       712 (22.8%)
 Neha       █████████████         624 (20.0%)
 Aman       ██████████            484 (15.5%)
 Karan      ███████               345 (11.0%)
 Vikas                             22 ( 0.7%)

 ACTIVITY HEATMAP (hour of day, columns 00 to 23)
           00    03    06    09    12    15    18    21    
 Rahul     . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒ 
 Priya     . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ . 
 Neha      . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░ 
 Aman      ▒ █ █ ▒ █ . . . . . . . . . . . . . . . . . . ▒  <- NIGHT OWL
 Karan     . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . . 
 Vikas     . . . . . . . ░ ▒ ░ ░ . ░ ▒ . ░ ░ █ ▒ ▒ ░ ░ ░ ▒ 

 THIS GROU

## Reflection
*What was the hardest part?*
Handling the datetime parsing and edge cases where participants went completely silent for days. Normalizing the archetype scores also took some thought to ensure every member received exactly one unique archetype fairly.

*What would you do differently?*
I would spend more time curating the custom stop-words list specific to hinglish context, and perhaps build a sentiment analyzer using basic word dictionaries.

*My Archetype:*
When running on my own chat, I was assigned **The Weekend Warrior** due to my heavy weekend activity!
